In [0]:
%run ../setup/config

In [0]:
%run ../setup/utils

In [0]:
movies_metadata_df = spark.read.format("delta").load(f"{silver_folder_path}/movies_metadata")
ratings_df = spark.read.format("delta").load(f"{silver_folder_path}/ratings")
links_df = spark.read.format("delta").load(f"{silver_folder_path}/links")
genres_df = spark.read.format("delta").load(f"{silver_folder_path}/movie_genres")

movies_metadata_df.printSchema()
ratings_df.printSchema()
links_df.printSchema()
genres_df.printSchema()

In [0]:
from pyspark.sql import functions as F

final_movies_df = (
    ratings_df.join(links_df, links_df.movie_id == ratings_df.movie_id, "inner")
    .join(movies_metadata_df, links_df.tmbd_id == movies_metadata_df.id, "inner")
    .groupBy(
        movies_metadata_df.id,
        "title",
        "overview",
        "release_date",
        "runtime",
        "original_language",
        "collection_name",
        "budget",
        "revenue",
    )
    .agg(
        F.avg("rating").alias("average_rating"),
        F.count("user_id").alias("number_of_ratings"),
    )
    .filter(F.col("release_date").isNotNull())
    .withColumn("release_year", F.year("release_date"))
    .withColumn("decade", (F.floor(F.col("release_year") / 10) * 10).cast("int"))
    .withColumn(
        "era",
        F.when(F.col("release_year") < 1970, F.lit("Classical (<1970)")).otherwise(
            F.lit("Modern (1970+)")
        ),
    )
    .withColumn(
        "gem_label",
        F.when(
            (F.col("release_year") < 1970)
            & (F.col("average_rating") >= 4.0)
            & (F.col("number_of_ratings") >= 50)
            & (F.col("number_of_ratings") <= 800),
            F.lit("Classical hidden gem"),
        )
        .when(
            (F.col("release_year") < 1970) & (F.col("average_rating") >= 4.0),
            F.lit("Famous classical (many ratings)"),
        )
        .when(
            F.col("release_year") < 1970,
            F.lit("Other classical"),
        )
        .otherwise(F.lit("Modern")),
    )
)

display(final_movies_df.orderBy(F.col("average_rating").desc()))

In [0]:
from pyspark.sql.window import Window

w_gems = Window.orderBy(
  F.col("average_rating").desc(),
  F.col("number_of_ratings").asc(),
)

classical_gems_df = (
  final_movies_df
    .filter(F.col("gem_label") == "Classical hidden gem")
    .withColumn("gem_rank", F.rank().over(w_gems))
    .orderBy("gem_rank")
)

display(classical_gems_df)

In [0]:
gems_with_genre_df = (
  classical_gems_df
    .join(genres_df, classical_gems_df.id == genres_df.id, "left")
    .groupBy(
      classical_gems_df.id,
      "title",
      "release_year",
      "decade",
      "runtime",
      "original_language",
      "average_rating",
      "number_of_ratings",
      "gem_rank",
    )
    .agg(F.min("genres_name").alias("sample_genre"))
    .orderBy("gem_rank")
)

display(gems_with_genre_df)

In [0]:
era_summary_df = (
  final_movies_df
    .filter(F.col("number_of_ratings") >= 20)
    .groupBy("era")
    .agg(
      F.count("*").alias("movies"),
      F.avg("average_rating").alias("mean_rating"),
      F.expr("percentile_approx(average_rating, 0.5)").alias("median_rating"),
      F.avg("number_of_ratings").alias("avg_rating_count"),
    )
    .orderBy(F.col("median_rating").desc())
)

display(era_summary_df)

In [0]:
import plotly.express as px

top_gems_pdf = gems_with_genre_df.limit(20).toPandas()

fig_gems = px.bar(
  top_gems_pdf,
  x="average_rating",
  y="title",
  orientation="h",
  color="decade",
  hover_data=["release_year", "number_of_ratings", "sample_genre", "original_language", "runtime"],
  text=top_gems_pdf["average_rating"].round(2),
  title="Top 20 classical hidden gems (pre-1970, high rating, 50–800 votes)",
  labels={
    "average_rating": "Average rating",
    "title": "Film",
    "decade": "Decade",
  },
  range_x=[3.9, 4.6],
)
fig_gems.update_traces(textposition="outside", cliponaxis=False)
fig_gems.update_layout(
  yaxis={"categoryorder": "total ascending"},
  height=700,
  margin=dict(r=80),
)
fig_gems.show()

In [0]:
# Popularity vs quality for classical films — gems sit top-leftish (high rating, fewer votes)
classical_scatter_pdf = (
  final_movies_df
    .filter(F.col("era") == "Classical (<1970)")
    .filter(F.col("number_of_ratings") >= 20)
    .orderBy(F.col("number_of_ratings").desc())
    .limit(800)
    .toPandas()
)

fig_scatter = px.scatter(
  classical_scatter_pdf,
  x="number_of_ratings",
  y="average_rating",
  color="gem_label",
  hover_name="title",
  hover_data=["release_year", "decade", "runtime"],
  title="Classical films: fame (rating count) vs quality — hidden gems highlighted",
  labels={
    "number_of_ratings": "Number of ratings (log)",
    "average_rating": "Average rating",
    "gem_label": "Label",
  },
  log_x=True,
  range_y=[2.5, 4.6],
  color_discrete_map={
    "Classical hidden gem": "#F4D35E",
    "Famous classical (many ratings)": "#5BC0BE",
    "Other classical": "#6C757D",
  },
)
fig_scatter.update_layout(height=550)
fig_scatter.show()

In [0]:
era_pdf = era_summary_df.toPandas()

fig_era = px.bar(
  era_pdf,
  x="era",
  y=["median_rating", "mean_rating"],
  barmode="group",
  hover_data=["movies", "avg_rating_count"],
  title="Classical vs modern — median/mean MovieLens rating (≥20 ratings per film)",
  labels={"value": "Rating", "era": "Era", "variable": "Metric"},
  range_y=[2.8, 3.6],
)
fig_era.for_each_trace(
  lambda t: t.update(name={"median_rating": "Median", "mean_rating": "Mean"}.get(t.name, t.name))
)
fig_era.update_layout(height=450)
fig_era.show()